In [2]:
from pathlib import Path
import pickle
import pandas as pd
import numpy as np
import os
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from config.config import TOI, TIME_WINDOWS
from config.paths import ERPAC_DIR, PCA_ERPAC_DIR

In [3]:
ERPAC_INPUT_DIR = Path(r"F:\# study 2\eeg_data\erpac\PCA\erpac_roi_tc")
LMM_OUTPUT_DIR = ERPAC_INPUT_DIR / "LMM"
LMM_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def find_erpac_files():
    files = sorted(ERPAC_INPUT_DIR.rglob("*_pca_roi_tc_erpac_results.pkl"))
    if not files:
        raise FileNotFoundError(f"No ERPAC .pkl files found under:\n{ERPAC_INPUT_DIR}")
    print(f"Found {len(files)} ERPAC files.")
    return files


def parse_condition(file_path, sub):
    suffix = "_pca_roi_tc_erpac_results.pkl"
    core = file_path.name[:-len(suffix)]
    condition_string = core[len(f"{sub}_"):]
    parts = condition_string.split("_")

    task, stage = parts[0], parts[1]
    block, adaptation_split = "NA", "NA"

    if task == "FTT":
        condition = "FTT"

    elif task == "DeCRAT":
        if len(parts) >= 3:
            block = parts[2]

        if block == "adaptation" and len(parts) >= 4:
            adaptation_split = parts[3]

        if block == "baseline":
            condition = "DeCRAT_baseline"
        elif block == "adaptation":
            condition = (
                f"DeCRAT_adaptation_{adaptation_split}"
                if adaptation_split != "NA"
                else "DeCRAT_adaptation"
            )
        else:
            condition = f"DeCRAT_{block}"

    else:
        raise ValueError(f"Unknown task in filename:\n{file_path.name}")

    return {
        "task": task,
        "task_stage": stage,
        "task_block": block,
        "adaptation_split": adaptation_split,
        "condition": condition,
    }


def average_erpac_map(erpac, times, tmin, tmax):
    erpac = np.asarray(erpac, dtype=float).squeeze()

    if erpac.ndim == 1:
        erpac = erpac[np.newaxis, :]

    times = np.asarray(times, dtype=float)

    if erpac.shape[-1] != len(times):
        raise ValueError("ERPAC and times dimensions do not match.")

    # Non-overlapping time windows: [start, end)
    mask = (times >= tmin - 1e-9) & (times < tmax - 1e-9)

    if not np.any(mask):
        raise ValueError(
            f"No samples found between {tmin} and {tmax} s"
        )

    # Remove NaN and infinite values before averaging
    values = erpac[..., mask].ravel()
    values = values[np.isfinite(values)]

    if values.size == 0:
        return np.nan

    return float(np.mean(values))


def build_lmm_dataframe(TIME_WINDOWS):
    rows = []
    files = find_erpac_files()

    for i, file_path in enumerate(files, start=1):
        sub = file_path.parent.name
        group = file_path.parent.parent.name
        meta = parse_condition(file_path, sub)

        task = meta["task"]
        stage = meta["task_stage"]

        print(
            f"[{i}/{len(files)}] {group} | {sub} | "
            f"{task} | {stage} | {meta['condition']}"
        )

        with open(file_path, "rb") as f:
            erpac_results = pickle.load(f)

        for coupling, roi_results in erpac_results.items():
            for roi, result in roi_results.items():

                erpac = result["erpac"]
                times = result["times"]

                # Calculate mean ERPAC for each time window
                for time_win, (tmin, tmax) in TIME_WINDOWS[task][stage].items():

                    mean_erpac = average_erpac_map(
                        erpac=erpac,
                        times=times,
                        tmin=tmin,
                        tmax=tmax,
                    )

                    rows.append({
                        "sub": sub,
                        "group": group,
                        "task": task,
                        "task_stage": stage,
                        "task_block": meta["task_block"],
                        "adaptation_split": meta["adaptation_split"],
                        "condition": meta["condition"],
                        "coupling": coupling,
                        "roi": roi,
                        "time_win": time_win,
                        "mean_erpac": mean_erpac,
                        "n_epochs": result.get("n_epochs", np.nan),
                        "pca_variance_threshold": result.get(
                            "pca_variance_threshold", np.nan
                        ),
                        "pca_n_components_retained": result.get(
                            "pca_n_components_retained", np.nan
                        ),
                        "pca_actual_variance_retained": result.get(
                            "pca_actual_variance_retained", np.nan
                        ),
                    })

        del erpac_results

    return pd.DataFrame(rows)


# Run
lmm_df = build_lmm_dataframe(TIME_WINDOWS)

print(lmm_df.head())
print(f"\nShape: {lmm_df.shape}")

# Check participant numbers per model cell
cell_counts = (
    lmm_df.groupby(
        [
            "group",
            "condition",
            "task_stage",
            "coupling",
            "roi",
            "time_win"
        ],
        observed=True
    )["sub"]
    .nunique()
)

print("\nSubjects per cell:")
print(cell_counts)

# Check missing values
print("\nMissing mean_erpac values:")
print(lmm_df["mean_erpac"].isna().sum())

# Check that each combination has three time windows
window_counts = (
    lmm_df.groupby(
        [
            "sub",
            "group",
            "condition",
            "task_stage",
            "coupling",
            "roi"
        ]
    )["time_win"]
    .nunique()
)

assert (window_counts == 3).all(), (
    "Some combinations do not have exactly three time windows."
)

# Save
output_path = (
    LMM_OUTPUT_DIR / "pca_erpac_LMM_all_tasks_time_windows.csv"
)

lmm_df.to_csv(output_path, index=False)

print(f"\nSaved to:\n{output_path}")



Found 376 ERPAC files.
[1/376] O | s1_pac_sub12 | DeCRAT | go | DeCRAT_adaptation_early
[2/376] O | s1_pac_sub12 | DeCRAT | go | DeCRAT_adaptation_late
[3/376] O | s1_pac_sub12 | DeCRAT | go | DeCRAT_baseline
[4/376] O | s1_pac_sub12 | DeCRAT | plan | DeCRAT_adaptation_early
[5/376] O | s1_pac_sub12 | DeCRAT | plan | DeCRAT_adaptation_late
[6/376] O | s1_pac_sub12 | DeCRAT | plan | DeCRAT_baseline
[7/376] O | s1_pac_sub12 | FTT | go | FTT
[8/376] O | s1_pac_sub12 | FTT | plan | FTT
[9/376] O | s1_pac_sub14 | DeCRAT | go | DeCRAT_adaptation_early
[10/376] O | s1_pac_sub14 | DeCRAT | go | DeCRAT_adaptation_late
[11/376] O | s1_pac_sub14 | DeCRAT | go | DeCRAT_baseline
[12/376] O | s1_pac_sub14 | DeCRAT | plan | DeCRAT_adaptation_early
[13/376] O | s1_pac_sub14 | DeCRAT | plan | DeCRAT_adaptation_late
[14/376] O | s1_pac_sub14 | DeCRAT | plan | DeCRAT_baseline
[15/376] O | s1_pac_sub14 | FTT | go | FTT
[16/376] O | s1_pac_sub14 | FTT | plan | FTT
[17/376] O | s1_pac_sub15 | DeCRAT | go | 